# script for investigating met office AWS data

- data from https://registry.opendata.aws/met-office-uk-land-observations/
- last 7 days (168 hrs) of data
- frequency: minutely


In [7]:
import boto3
import pandas as pd
from io import BytesIO
from botocore import UNSIGNED
from botocore.config import Config

In [8]:
import boto3 # may need to set up aws credentials?
from botocore import UNSIGNED
from botocore.config import Config


BUCKET_NAME = "met-office-land-observations-data"
# ^ most important bit is the bucket name, which is public and can be found in the met office docs

s3 = boto3.client(
    "s3",
    region_name="eu-west-2",
    config=Config(signature_version=UNSIGNED), # no authentication, public. otherwise boto3 expects aws creds
)

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    MaxKeys=10, # only show 10 items for now
)

for item in response.get("Contents", []):
    print(item["Key"])

202609112359_202609112234_202609112259/herstmonceux_west_end_automatic_weather_station_f4458904-0dee-4b35-a3e0-a88a3badc337.csv
202609120059_202609112300_202609112359/herstmonceux_west_end_automatic_weather_station_6377cf3c-991a-40fd-acdc-3da1f28e70f4.csv
202609120159_202609120000_202609120059/herstmonceux_west_end_automatic_weather_station_07f1e2a5-d825-4778-8fe3-daf57e4a7f6b.csv
202609120259_202609120100_202609120159/herstmonceux_west_end_automatic_weather_station_cb1fd297-de86-437b-abd8-af2dd4348cd5.csv
202609120359_202609120200_202609120259/herstmonceux_west_end_automatic_weather_station_16cc0bca-3ce5-4b77-9fd7-c1b079a1d2dc.csv
202609120459_202609120300_202609120359/herstmonceux_west_end_automatic_weather_station_edd9210f-ef5d-4601-9abf-ff6123311735.csv
202609120559_202609120400_202609120459/herstmonceux_west_end_automatic_weather_station_0d3dc83e-0ae5-44aa-bc7c-453a772fe2f0.csv
202609120659_202609120500_202609120559/herstmonceux_west_end_automatic_weather_station_bd7df868-2e7d-4d6

In [9]:
first_key = response["Contents"][0]["Key"]

print(first_key)

202609112359_202609112234_202609112259/herstmonceux_west_end_automatic_weather_station_f4458904-0dee-4b35-a3e0-a88a3badc337.csv


In [13]:
response

{'ResponseMetadata': {'RequestId': 'GQXPR4BFK4HDYKB9',
  'HostId': 'isrw5aQhix7Lb1/A9w15XTDrBHfN9hxkwfJu8YtiintQQrOzoDjFByWvb3r9I3Dy/FhboA3RswM=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'isrw5aQhix7Lb1/A9w15XTDrBHfN9hxkwfJu8YtiintQQrOzoDjFByWvb3r9I3Dy/FhboA3RswM=',
   'x-amz-request-id': 'GQXPR4BFK4HDYKB9',
   'date': 'Thu, 24 Sep 2026 13:19:55 GMT',
   'x-amz-bucket-region': 'eu-west-2',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'IsTruncated': True,
 'Contents': [{'Key': '202609112359_202609112234_202609112259/herstmonceux_west_end_automatic_weather_station_f4458904-0dee-4b35-a3e0-a88a3badc337.csv',
   'LastModified': datetime.datetime(2026, 9, 17, 0, 0, 49, tzinfo=tzutc()),
   'ETag': '"46ff7b3333f7cf8e432eceeccf60d31a"',
   'ChecksumAlgorithm': ['CRC32'],
   'ChecksumType': 'FULL_OBJECT',
   'Size': 24082,
   'StorageClass': 'STANDARD'},
  {'Key': '202609120059_202609112300_20260911235

In [14]:
csv_response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=first_key,
)

In [15]:
csv_response.keys()

dict_keys(['ResponseMetadata', 'AcceptRanges', 'Expiration', 'LastModified', 'ContentLength', 'ETag', 'ChecksumCRC32', 'ChecksumType', 'VersionId', 'ContentType', 'ServerSideEncryption', 'Metadata', 'ReplicationStatus', 'Body'])

In [16]:
raw_data = csv_response["Body"].read()

print(raw_data[:2000].decode("utf-8"))

timestep|name|longitude|latitude|accumulated_precipitation_1_minute_total|accumulated_precipitation_1_minute_total_qc|air_pressure_near_surface_1_minute_mean|air_pressure_near_surface_1_minute_mean_qc|air_pressure_near_surface_mean_sea_level_1_minute_mean|air_pressure_near_surface_mean_sea_level_1_minute_mean_qc|air_temperature_near_surface_1_minute_mean|air_temperature_near_surface_1_minute_mean_qc|air_temperature_near_surface_secondary_1_minute_mean|air_temperature_near_surface_secondary_1_minute_mean_qc|air_temperature_near_surface_sensor2_1_minute_mean|air_temperature_near_surface_sensor2_1_minute_mean_qc|cloud_base_height_layer1_1_minute_30_minute_rolling_min|cloud_base_height_layer1_1_minute_30_minute_rolling_min_qc|cloud_base_height_layer2_1_minute_30_minute_rolling_min|cloud_base_height_layer2_1_minute_30_minute_rolling_min_qc|cloud_base_height_layer3_1_minute_30_minute_rolling_min|cloud_base_height_layer3_1_minute_30_minute_rolling_min_qc|cloud_cover_layer1_1_minute_30_minute_

In [ ]:
import pandas as pd
from io import BytesIO

df = pd.read_csv(BytesIO(raw_data))

# this code fails because of the vertical bar delimiter

ParserError: Error tokenizing data. C error: Expected 2 fields in line 13, saw 3


Vertical bar is delimiter, while pd.read.csv assumes commas by default. So, fix this

In [20]:
df = pd.read_csv(BytesIO(raw_data), sep="|")

In [21]:
df.head()

,timestep,name,longitude,latitude,accumulated_precipitation_1_minute_total,accumulated_precipitation_1_minute_total_qc,air_pressure_near_surface_1_minute_mean,air_pressure_near_surface_1_minute_mean_qc,air_pressure_near_surface_mean_sea_level_1_minute_mean,air_pressure_near_surface_mean_sea_level_1_minute_mean_qc,...,soil_temperature_100cm_1_minute_mean,soil_temperature_100cm_1_minute_mean_qc,soil_temperature_10cm_1_minute_mean,soil_temperature_10cm_1_minute_mean_qc,soil_temperature_30cm_1_minute_mean,soil_temperature_30cm_1_minute_mean_qc,wind_direction_near_surface_1_minute_mean,wind_direction_near_surface_1_minute_mean_qc,wind_speed_near_surface_1_minute_mean,wind_speed_near_surface_1_minute_mean_qc
0,2026-09-11T22:34:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",1017.780,"{""good"": []}",NaN,NaN,...,17.212471,"{""good"": []}",15.921395,"{""good"": []}",17.799696,"{""good"": []}",23.030426,"{""good"": []}",0.160450,"{""good"": []}"
1,2026-09-11T22:35:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",1017.775,"{""good"": []}",NaN,NaN,...,17.214383,"{""good"": []}",15.917620,"{""good"": []}",17.801059,"{""good"": []}",85.698765,"{""good"": []}",1.142883,"{""good"": []}"
2,2026-09-11T22:36:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",1017.775,"{""suspect"": [""2 case(s) of \""missing previous ...",NaN,NaN,...,17.214236,"{""suspect"": [""2 case(s) of \""missing previous ...",15.909513,"{""suspect"": [""2 case(s) of \""missing previous ...",17.801613,"{""suspect"": [""2 case(s) of \""missing previous ...",84.861667,"{""good"": []}",0.803092,"{""suspect"": [""2 case(s) of \""missing previous ..."
3,2026-09-11T22:37:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",1017.760,"{""good"": []}",NaN,NaN,...,17.214183,"{""good"": []}",15.898833,"{""good"": []}",17.798131,"{""good"": []}",83.691471,"{""good"": []}",0.889208,"{""good"": []}"
4,2026-09-11T22:38:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",1017.770,"{""good"": []}",NaN,NaN,...,17.212621,"{""good"": []}",15.891681,"{""good"": []}",17.801312,"{""good"": []}",79.402005,"{""good"": []}",1.456750,"{""good"": []}"


^ many NaNs

In [22]:
df.shape

(26, 66)

In [23]:
df.columns.tolist()

['timestep',
 'name',
 'longitude',
 'latitude',
 'accumulated_precipitation_1_minute_total',
 'accumulated_precipitation_1_minute_total_qc',
 'air_pressure_near_surface_1_minute_mean',
 'air_pressure_near_surface_1_minute_mean_qc',
 'air_pressure_near_surface_mean_sea_level_1_minute_mean',
 'air_pressure_near_surface_mean_sea_level_1_minute_mean_qc',
 'air_temperature_near_surface_1_minute_mean',
 'air_temperature_near_surface_1_minute_mean_qc',
 'air_temperature_near_surface_secondary_1_minute_mean',
 'air_temperature_near_surface_secondary_1_minute_mean_qc',
 'air_temperature_near_surface_sensor2_1_minute_mean',
 'air_temperature_near_surface_sensor2_1_minute_mean_qc',
 'cloud_base_height_layer1_1_minute_30_minute_rolling_min',
 'cloud_base_height_layer1_1_minute_30_minute_rolling_min_qc',
 'cloud_base_height_layer2_1_minute_30_minute_rolling_min',
 'cloud_base_height_layer2_1_minute_30_minute_rolling_min_qc',
 'cloud_base_height_layer3_1_minute_30_minute_rolling_min',
 'cloud_base_

##### interesting; the file schema contains both the measurements and the qc field (..._qc")

In [24]:
df.dtypes

timestep                                            str
name                                                str
longitude                                       float64
latitude                                        float64
accumulated_precipitation_1_minute_total        float64
                                                 ...   
soil_temperature_30cm_1_minute_mean_qc              str
wind_direction_near_surface_1_minute_mean       float64
wind_direction_near_surface_1_minute_mean_qc        str
wind_speed_near_surface_1_minute_mean           float64
wind_speed_near_surface_1_minute_mean_qc            str
Length: 66, dtype: object

In [25]:
df[["timestep", "name", "longitude", "latitude", "global_radiation_1_minute_mean"]].head(10)

,timestep,name,longitude,latitude,global_radiation_1_minute_mean
0,2026-09-11T22:34:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
1,2026-09-11T22:35:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
2,2026-09-11T22:36:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
3,2026-09-11T22:37:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
4,2026-09-11T22:38:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
5,2026-09-11T22:39:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
6,2026-09-11T22:40:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
7,2026-09-11T22:41:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
8,2026-09-11T22:42:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0
9,2026-09-11T22:43:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0


Let's narrow the df to solar vars:


In [26]:
solar_cols = [
    "timestep",
    "name",
    "longitude",
    "latitude",
    "global_radiation_1_minute_mean",
    "global_radiation_1_minute_mean_qc",
    "direct_solar_irradiance_1_minute_mean",
    "direct_solar_irradiance_1_minute_mean_qc",
]

df[solar_cols].head(10)

,timestep,name,longitude,latitude,global_radiation_1_minute_mean,global_radiation_1_minute_mean_qc,direct_solar_irradiance_1_minute_mean,direct_solar_irradiance_1_minute_mean_qc
0,2026-09-11T22:34:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
1,2026-09-11T22:35:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
2,2026-09-11T22:36:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
3,2026-09-11T22:37:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
4,2026-09-11T22:38:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
5,2026-09-11T22:39:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
6,2026-09-11T22:40:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
7,2026-09-11T22:41:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
8,2026-09-11T22:42:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"
9,2026-09-11T22:43:00Z,"Herstmonceux, West End Automatic Weather Station",0.32,50.89,0.0,"{""good"": []}",0.0,"{""good"": []}"


In [27]:
df["global_radiation_1_minute_mean"].notna().sum()

np.int64(26)

^ this station doesn't have the global radiation data

In [28]:
df.notna().sum().sort_values(ascending=False)

timestep                                                   26
precipitation_intensity_1_minute_rolling_algorithm_qc      26
name                                                       26
direct_solar_irradiance_1_minute_mean                      26
direct_solar_irradiance_1_minute_mean_qc                   26
                                                           ..
air_temperature_near_surface_sensor2_1_minute_mean_qc       0
dew_point_temperature_1_minute_mean_qc                      0
cloud_cover_layer2_1_minute_30_minute_weighted_mean         0
cloud_base_height_layer2_1_minute_30_minute_rolling_min     0
dew_point_temperature_1_minute_mean                         0
Length: 66, dtype: int64

Now let's try to answer the more general Q of what stations have this solar data (most recently at least)

In [34]:
first_key

'202609112359_202609112234_202609112259/herstmonceux_west_end_automatic_weather_station_f4458904-0dee-4b35-a3e0-a88a3badc337.csv'

In [37]:
batch_prefix = first_key.split('/',1)[0]

In [38]:
batch_prefix

'202609112359_202609112234_202609112259'

In [39]:
# this removed the slash, so need to add it back:
batch_prefix = batch_prefix + '/'

In [40]:
batch_prefix

'202609112359_202609112234_202609112259/'

In [41]:
batch_response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=batch_prefix,
)

batch_keys = [
    item["Key"]
    for item in batch_response.get("Contents", [])
]

print(f"Found {len(batch_keys)} files")

Found 1 files


In [42]:
from io import BytesIO
import pandas as pd

results = []

for key in batch_keys:
    csv_response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key,
    )

    raw_data = csv_response["Body"].read()

    station_df = pd.read_csv(
        BytesIO(raw_data),
        sep="|",
    )

    results.append({
        "name": station_df["name"].iloc[0],
        "latitude": station_df["latitude"].iloc[0],
        "longitude": station_df["longitude"].iloc[0],
        "global_radiation_count": station_df["global_radiation_1_minute_mean"].notna().sum(),
        "direct_irradiance_count": station_df["direct_solar_irradiance_1_minute_mean"].notna().sum(),
    })

In [43]:
availability = pd.DataFrame(results)

availability.head()

,name,latitude,longitude,global_radiation_count,direct_irradiance_count
0,"Herstmonceux, West End Automatic Weather Station",50.89,0.32,26,26


In [44]:
solar_stations = availability[
    (availability["global_radiation_count"] > 0)
    | (availability["direct_irradiance_count"] > 0)
]

solar_stations

,name,latitude,longitude,global_radiation_count,direct_irradiance_count
0,"Herstmonceux, West End Automatic Weather Station",50.89,0.32,26,26


In [45]:
print(f"Total stations in batch: {len(availability)}")
print(f"Stations with solar data: {len(solar_stations)}")
print(f"Stations with global radiation: {(availability['global_radiation_count'] > 0).sum()}")
print(f"Stations with direct irradiance: {(availability['direct_irradiance_count'] > 0).sum()}")

Total stations in batch: 1
Stations with solar data: 1
Stations with global radiation: 1
Stations with direct irradiance: 1


Didn't need to do all this to figure this out, but: the hardcoded batch_prefix is too limiting; we're not getting all the stations. So let's be more systematic...

In [46]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    MaxKeys=1000,
)

keys = [
    item["Key"]
    for item in response.get("Contents", [])
]

print(f"Objects returned: {len(keys)}")
print(f"More objects available: {response['IsTruncated']}")

Objects returned: 1000
More objects available: True


In [47]:
# now let's see how many different prefixes those first 1000 objects have:
prefixes = [key.split("/")[0] for key in keys]

print(f"Unique prefixes: {len(set(prefixes))}") # set() useful beacuse it removes duplicates, so we can count unique prefixes

Unique prefixes: 125


In [48]:
from collections import Counter

prefix_counts = Counter(prefixes)

prefix_counts.most_common(20)

[('202609160059_202609152300_202609152359', 250),
 ('202609160159_202609160000_202609160059', 250),
 ('202609160259_202609160100_202609160159', 183),
 ('202609152359_202609152200_202609152259', 74),
 ('202609152159_202609152000_202609152059', 20),
 ('202609152059_202609151900_202609151959', 17),
 ('202609152259_202609152100_202609152159', 15),
 ('202609151759_202609151600_202609151659', 14),
 ('202609151959_202609151800_202609151859', 12),
 ('202609150459_202609150300_202609150359', 11),
 ('202609151859_202609151700_202609151759', 11),
 ('202609150559_202609150400_202609150459', 7),
 ('202609150659_202609150500_202609150559', 6),
 ('202609150759_202609150600_202609150659', 6),
 ('202609151659_202609151500_202609151559', 6),
 ('202609151059_202609150900_202609150959', 3),
 ('202609151459_202609151300_202609151359', 3),
 ('202609151559_202609151400_202609151459', 3),
 ('202609150159_202609150000_202609150059', 2),
 ('202609150859_202609150700_202609150759', 2)]

- prefixes look hourly
- each prefix contains only 11 files, so a prefix is likely not the hourly batch for every station. Instead, seems to be grouped accorind to when data were released after processing
- abandon idea of selecting a prefix. instead ask: across all station files available in the bucket, which statations have reported solar data?
- need pagination since S3 only returns 1000 objects per request. boto3 has a paginator

In [49]:
paginator = s3.get_paginator("list_objects_v2")

pages = paginator.paginate(
    Bucket=BUCKET_NAME
)
# ^ not stored in memory, but something python can iterate through, one S3 response page at a time

In [50]:
# now collect all the keys:
all_keys = []

for page in pages:
    for item in page.get("Contents", []):
        all_keys.append(item["Key"])

print(f"Total objects in bucket: {len(all_keys)}")

Total objects in bucket: 52801


In [51]:
filenames = [
    key.split("/")[-1]
    for key in all_keys
]

print(f"Total files: {len(filenames)}")
print(f"Unique filenames: {len(set(filenames))}")

Total files: 52801
Unique filenames: 52801


In [52]:
all_keys[:10]

['202609112359_202609112234_202609112259/herstmonceux_west_end_automatic_weather_station_f4458904-0dee-4b35-a3e0-a88a3badc337.csv',
 '202609120059_202609112300_202609112359/herstmonceux_west_end_automatic_weather_station_6377cf3c-991a-40fd-acdc-3da1f28e70f4.csv',
 '202609120159_202609120000_202609120059/herstmonceux_west_end_automatic_weather_station_07f1e2a5-d825-4778-8fe3-daf57e4a7f6b.csv',
 '202609120259_202609120100_202609120159/herstmonceux_west_end_automatic_weather_station_cb1fd297-de86-437b-abd8-af2dd4348cd5.csv',
 '202609120359_202609120200_202609120259/herstmonceux_west_end_automatic_weather_station_16cc0bca-3ce5-4b77-9fd7-c1b079a1d2dc.csv',
 '202609120459_202609120300_202609120359/herstmonceux_west_end_automatic_weather_station_edd9210f-ef5d-4601-9abf-ff6123311735.csv',
 '202609120559_202609120400_202609120459/herstmonceux_west_end_automatic_weather_station_0d3dc83e-0ae5-44aa-bc7c-453a772fe2f0.csv',
 '202609120659_202609120500_202609120559/herstmonceux_west_end_automatic_wea

In [53]:
all_keys[-10:]

['202609241359_202609241200_202609241259/wight_st_catherines_point_automatic_weather_station_87776c7e-fecf-4621-a9ed-7dd984f04af4.csv',
 '202609241359_202609241200_202609241259/winchcombe_sudeley_castle_automatic_weather_station_cb22bcff-fb18-4c17-929d-0b3fafc4d942.csv',
 '202609241359_202609241200_202609241259/winterbourne_no_2_automatic_weather_station_b8342818-fda0-40c4-8c8a-fdb782a9b25e.csv',
 '202609241359_202609241200_202609241259/wisley_automatic_weather_station_be2859db-e3b6-49eb-bbba-ecd8fb30146d.csv',
 '202609241359_202609241200_202609241259/wittering_automatic_weather_station_3067ab70-dabb-40b0-88fe-77ed4e33b304.csv',
 '202609241359_202609241200_202609241259/woburn_automatic_weather_station_9852dd99-5e90-4d07-ad42-cf378c89c730.csv',
 '202609241359_202609241200_202609241259/writtle_automatic_weather_station_32435731-af14-43a8-8b18-def7a759d54d.csv',
 '202609241359_202609241200_202609241259/wych_cross_automatic_weather_station_54bac7e2-e9b5-46d8-a681-5ddc6b8801cc.csv',
 '20260

^ i did this at 2:04pm on 2026/09/11 so the latest is at 11:59UTC same day, so basically 12noon UTC or 1pm BST time, so ~65mins latency there for most recent data? but that's the time of release, maybe not the time of data collected...

Now let's take a look at all files on 2026/09/10 (yesterday, given the day I'm writing), at 12noon UTC (note I'm in BST time which is UTC+1)

In [59]:
midday_keys = []

for key in all_keys:
    prefix = key.split("/")[0]
    parts = prefix.split("_")

    if len(parts) == 3:
        observation_start = parts[1]

        if observation_start.startswith("2026092312"):
            midday_keys.append(key)

print(f"Files matching target hour: {len(midday_keys)}")

Files matching target hour: 262


^ updated this to sept 23 when i redid it; ideally shouldn't be hardcoded but this is just for diagnostics anyway

In [60]:
midday_keys[:10]

['202609231359_202609231200_202609231259/aberdaron_automatic_weather_station_070756f8-b536-4778-a39d-3d1cee82a041.csv',
 '202609231359_202609231200_202609231259/abergwyngregyn_automatic_weather_station_637b523e-fa24-4b2a-9c6d-34c4134a479e.csv',
 '202609231359_202609231200_202609231259/aberporth_automatic_weather_station_2a95065c-033d-4360-8c42-0def42ad981e.csv',
 '202609231359_202609231200_202609231259/aboyne_no_2_automatic_weather_station_a6b73120-8f0f-4546-9059-44dbf322f360.csv',
 '202609231359_202609231200_202609231259/achnagart_automatic_weather_station_f78f57e0-ef1b-4e8b-9182-3bd52fa1d5b7.csv',
 '202609231359_202609231200_202609231259/akrotiri_cyprus_automatic_weather_station_a364f3df-4820-49c9-a097-e63ab13d6e62.csv',
 '202609231359_202609231200_202609231259/albemarle_automatic_weather_station_94dcb26a-078e-4ba7-a7f7-d6554223a4f4.csv',
 '202609231359_202609231200_202609231259/aldergrove_automatic_weather_station_c443bbf3-b8f4-4908-9bae-4b68ab52537a.csv',
 '202609231359_20260923120

In [61]:
solar_results = []

for key in midday_keys:
    csv_response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key,
    )

    raw_data = csv_response["Body"].read()

    station_df = pd.read_csv(
        BytesIO(raw_data),
        sep="|",
    )

    global_count = station_df["global_radiation_1_minute_mean"].notna().sum()
    direct_count = station_df["direct_solar_irradiance_1_minute_mean"].notna().sum()

    solar_results.append({
        "name": station_df["name"].iloc[0],
        "latitude": station_df["latitude"].iloc[0],
        "longitude": station_df["longitude"].iloc[0],
        "global_count": global_count,
        "direct_count": direct_count,
    })

solar_availability = pd.DataFrame(solar_results)

In [62]:
len(solar_availability) # how many stations have data at this hour? (it only needs to have one row of data to be included in the list, even if it has no solar data, i think)

262

In [63]:
(solar_availability["global_count"] > 0).sum() # how many stations have global radiation data at this hour?

np.int64(87)

In [64]:
(solar_availability["direct_count"] > 0).sum()

np.int64(100)

^ answers: 83 have GHI, 94 have DNI. That's not too bad! probably more then the number of solarsense nodes I'd realistically deploy in my dphil...

edit on 24 sept: now i'm getting 87 GHI, 100 DNI. So it seems to vary quite a bit day by day.

In [65]:
#which ones?
solar_stations = solar_availability[
    (solar_availability["global_count"] > 0)
    | (solar_availability["direct_count"] > 0)
].copy()

solar_stations.sort_values("name")

,name,latitude,longitude,global_count,direct_count
0,Aberdaron Automatic Weather Station,52.79,-4.74,60,60
1,Abergwyngregyn Automatic Weather Station,53.24,-4.01,60,60
2,Aberporth Automatic Weather Station,52.14,-4.57,60,60
5,"Akrotiri, Cyprus Automatic Weather Station",34.58,32.98,0,60
7,Aldergrove Automatic Weather Station,54.66,-6.23,60,60
...,...,...,...,...,...
251,Winterbourne No 2 Automatic Weather Station,52.46,-1.93,60,60
252,Wisley Automatic Weather Station,51.31,-0.48,60,60
253,Wittering Automatic Weather Station,52.61,-0.47,60,60
254,Woburn Automatic Weather Station,52.01,-0.60,60,60


In [74]:
solar_stations

,name,latitude,longitude,global_count,direct_count
0,Aberdaron Automatic Weather Station,52.79,-4.74,60,60
1,Abergwyngregyn Automatic Weather Station,53.24,-4.01,60,60
2,Aberporth Automatic Weather Station,52.14,-4.57,60,60
5,"Akrotiri, Cyprus Automatic Weather Station",34.58,32.98,0,60
7,Aldergrove Automatic Weather Station,54.66,-6.23,60,60
...,...,...,...,...,...
251,Winterbourne No 2 Automatic Weather Station,52.46,-1.93,60,60
252,Wisley Automatic Weather Station,51.31,-0.48,60,60
253,Wittering Automatic Weather Station,52.61,-0.47,60,60
254,Woburn Automatic Weather Station,52.01,-0.60,60,60


In [66]:
2

2

In [ ]:
import folium

m = folium.Map(
    location=[54.5, -3],
    zoom_start=5,
)

for _, station in solar_stations.iterrows():
    folium.Marker(
        location=[station["latitude"], station["longitude"]],
        tooltip=station["name"],
    ).add_to(m)

# could have used 'for i,j in solar_stations' but since we don't use i in the loop, the underscore is the convention. 
# but if you open up solar_stations, you'll see that there is an initial column seemingly with no title, and then the second
# column title is 'name', so I guess that's why we need a dummy variable there. The underscore is not special python syntax;
# it's just a normal variable name, used by convention to show that a value will not be used!

# m


In [80]:
solar_stations

,name,latitude,longitude,global_count,direct_count
0,Aberdaron Automatic Weather Station,52.79,-4.74,60,60
1,Abergwyngregyn Automatic Weather Station,53.24,-4.01,60,60
2,Aberporth Automatic Weather Station,52.14,-4.57,60,60
5,"Akrotiri, Cyprus Automatic Weather Station",34.58,32.98,0,60
7,Aldergrove Automatic Weather Station,54.66,-6.23,60,60
...,...,...,...,...,...
251,Winterbourne No 2 Automatic Weather Station,52.46,-1.93,60,60
252,Wisley Automatic Weather Station,51.31,-0.48,60,60
253,Wittering Automatic Weather Station,52.61,-0.47,60,60
254,Woburn Automatic Weather Station,52.01,-0.60,60,60


In [70]:
m.save("../data/solar_stations_map.html")

In [71]:
import folium

m = folium.Map(
    location=[54.5, -3],
    zoom_start=5,
)

for _, station in solar_availability.iterrows():

    # Global horizontal irradiance (GHI)
    if station["global_count"] > 0:
        folium.CircleMarker(
            location=[station["latitude"], station["longitude"]],
            radius=6,
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.7,
            tooltip=f"{station['name']} — GHI",
        ).add_to(m)

    # Direct irradiance
    if station["direct_count"] > 0:
        folium.CircleMarker(
            location=[station["latitude"], station["longitude"]],
            radius=3,
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=0.9,
            tooltip=f"{station['name']} — Direct",
        ).add_to(m)

# m

In [72]:
m.save("../data/solar_stations_map.html")

Wow, there is one in gibraltar and one in cyrpus as well!
- also jersey, IoM, NI, and very northern scotland, but those are to be expected I guess